# Explore Madrid Accident Data

This notebook explores the raw CSV data from Madrid's Open Data portal to decide which columns should be stored in the new `accident_participants` table.

In [ ]:
import pandas as pd
import io
import requests
from pathlib import Path

# Sample URL for 2024 accidents
URL_2024 = "https://datos.madrid.es/dataset/300228-0-accidentes-trafico-detalle/resource/300228-2-accidentes-trafico-detalle-csv/download/300228-2-accidentes-trafico-detalle-csv.csv"

print("Downloading 2024 data...")
resp = requests.get(URL_2024)
df = pd.read_csv(io.BytesIO(resp.content), sep=";", encoding="utf-8-sig")

# Normalize columns
df.columns = [c.lower().strip().replace(" ", "_") for c in df.columns]

print(f"Total records: {len(df)}")
print(f"Unique accidents: {df['num_expediente'].nunique()}")
df.head()

## Column Analysis

Let's look at the available columns and some sample values for each.

In [ ]:
analysis = []
for col in df.columns:
    analysis.append({
        "Column": col,
        "Unique Values": df[col].nunique(),
        "Sample Values": ", ".join(map(str, df[col].dropna().unique()[:5])),
        "Nulls": df[col].isna().sum()
    })

pd.DataFrame(analysis)

## Typical participant-level columns

Based on the data, these columns vary per person for the same `num_expediente`:

In [ ]:
participant_cols = [
    "tipo_persona",
    "rango_edad",
    "sexo",
    "cod_lesividad",
    "lesividad",
    "tipo_vehiculo"
]

# Show an example of multiple participants in one accident
multi_participant_ids = df['num_expediente'].value_counts()
example_id = multi_participant_ids[multi_participant_ids > 1].index[0]

print(f"Example accident ID with multiple participants: {example_id}")
df[df['num_expediente'] == example_id][participant_cols]